# Geração de imagens sintéticas com Gemini

Este notebook gera **imagens sintéticas** para o dataset do **Urban Disaster Monitor**, usando o modelo **Gemini 2.5 Flash Image**. As imagens simulam cenários de desastre (inundações) com animais (vacas, cavalos, gatos, cães) em estilo documental/fotojornalístico, para enriquecer o conjunto de dados e reduzir desbalanceamento de classes.

---
**Projeto:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Autores:** Carolina Soares, João Galdino

## Opção 1: Uso manual (interface web)

1. Acesse [Gemini 2.5 Flash Image](https://ai.google.com/research/gemini).
2. Insira os prompts na seção de geração de imagens.
3. Ajuste resolução e estilo conforme necessário.
4. Gere e faça o download das imagens para inclusão no dataset.

## Opção 2: Geração em lote via API

As células abaixo usam a **API do Google Generative AI** para gerar várias imagens em sequência. É necessário uma chave de API (Google AI Studio ou Google Cloud).

### Configuração e dependências

Instale as bibliotecas:

In [ ]:
!pip install google-generativeai pillow

Obtenha uma chave de API em [Google AI Studio](https://aistudio.google.com/apikey) ou [Google Cloud Console](https://console.cloud.google.com/). Configure a variável de ambiente `GOOGLE_API_KEY` ou use o código abaixo com `getpass` para inserir a chave de forma segura.

### Geração das imagens

O código abaixo carrega os prompts, chama a API do Gemini e salva cada imagem na pasta de saída. Ajuste `OUTPUT_DIR` e a lista `prompts` conforme necessário.

In [ ]:
# Configurar API key: use variável de ambiente ou insira interativamente
import os
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    try:
        import getpass
        api_key = getpass.getpass("Cole sua GOOGLE_API_KEY e pressione Enter: ")
    except Exception:
        api_key = "SUA_CHAVE_AQUI"  # substitua pela sua chave para teste local
print("API key configurada." if api_key and api_key != "SUA_CHAVE_AQUI" else "Defina GOOGLE_API_KEY ou substitua SUA_CHAVE_AQUI.")

In [ ]:
import os
import google.generativeai as genai
from PIL import Image
from io import BytesIO
import time

genai.configure(api_key=api_key)

OUTPUT_DIR = "imagens_geradas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prompts em estilo documental/fotojornalístico para cenários de inundação (Urban Disaster Monitor)
prompts = [
    "Realistic low-resolution documentary photo of a cow isolated in floodwater reflecting sunset colors, authentic disaster photo",
    "Documentary style photo of a white horse trapped in a pasture with floodwater, overcast sky, realistic proportions, authentic disaster scene",
    "Low-resolution documentary photo of two wet cats stranded on a rooftop surrounded by floodwater, realistic proportions, visible faces, authentic urban flood event",
    "Documentary photo of a kitten on a balcony during flood, water rising near railing, cloudy weather, clear body visible, authentic disaster photo",
    "Photojournalism style disaster photo of a German shepherd swimming in a flooded street, cars partially submerged, realistic proportions, natural lighting",
    "Low quality documentary style photo of a stray dog on a small wooden raft during flood, rescue scene, realistic body intact",
    "Documentary flood photo of cows standing in a partially submerged field, water up to their legs, cloudy sky, authentic proportions",
    "Authentic documentary photo of a brown horse standing in a flooded street after heavy rain, realistic proportions, clear body visible, imperfect disaster photo",
]

# Nome do modelo de geração de imagens (consulte a documentação Google AI para versões atuais)
MODEL_NAME = "gemini-2.5-flash-image-preview"
MAX_RETRIES = 2
DELAY_BETWEEN_REQUESTS = 1.0  # segundos (evitar rate limit)

def generate_and_save_image(prompt, index, total):
    print(f"[{index}/{total}] {prompt[:60]}...")
    for attempt in range(MAX_RETRIES):
        try:
            model = genai.GenerativeModel(model_name=MODEL_NAME)
            response = model.generate_content(prompt)
            image_data = None
            if response.candidates:
                for part in response.candidates[0].content.parts:
                    if part.inline_data:
                        image_data = part.inline_data.data
                        break
            if image_data:
                image = Image.open(BytesIO(image_data))
                filename = os.path.join(OUTPUT_DIR, f"image_{index:04d}.png")
                image.save(filename)
                print(f"  → Salvo: {filename}")
                return True
            if response.candidates and response.candidates[0].content.parts:
                for part in response.candidates[0].content.parts:
                    if part.text:
                        print(f"  Modelo respondeu (texto): {part.text[:80]}...")
        except Exception as e:
            print(f"  Tentativa {attempt + 1}/{MAX_RETRIES} falhou: {e}")
        time.sleep(DELAY_BETWEEN_REQUESTS)
    print(f"  FALHA após {MAX_RETRIES} tentativas.")
    return False

for i, prompt in enumerate(prompts, start=1):
    generate_and_save_image(prompt, i, len(prompts))
    time.sleep(DELAY_BETWEEN_REQUESTS)

print("Geração concluída.")
print(f"Imagens em: {os.path.abspath(OUTPUT_DIR)}")

### Listar imagens geradas

Execute a célula abaixo para listar os arquivos na pasta de saída.

In [ ]:
import glob

files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
print(f"Total: {len(files)} imagens")

for f in files[:15]:
    print(f"  {os.path.basename(f)}")
    
if len(files) > 15:
    print(f"  ... e mais {len(files) - 15}")